In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, chi2
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Load dataset
data = pd.read_csv("amazon_uk_product_data.csv")
data.head()

# Part 1: Analyzing Best-Seller Trends Across Product Categories
# Crosstab Analysis
crosstab = pd.crosstab(data['product_category'], data['isBestSeller'], normalize='index')
crosstab['best_seller_proportion'] = crosstab[1]
crosstab = crosstab.sort_values(by='best_seller_proportion', ascending=False)
print(crosstab.head())

# Chi-square test
chi2_stat, p_val, dof, ex = chi2_contingency(pd.crosstab(data['product_category'], data['isBestSeller']))
print("Chi-square test p-value:", p_val)

# Cramér's V Calculation
cramers_v = np.sqrt(chi2_stat / (data.shape[0] * (min(crosstab.shape) - 1)))
print("Cramér's V:", cramers_v)

# Stacked Bar Chart
crosstab.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Best-Seller Status by Product Category')
plt.xlabel('Product Category')
plt.ylabel('Proportion')
plt.show()

# Part 2: Exploring Product Prices and Ratings Across Categories and Brands
# Preliminary Step: Remove outliers in product prices using IQR
Q1 = data['price'].quantile(0.25)
Q3 = data['price'].quantile(0.75)
IQR = Q3 - Q1
data_no_outliers = data[(data['price'] >= Q1 - 1.5 * IQR) & (data['price'] <= Q3 + 1.5 * IQR)]

# Violin Plot for Price Distribution Across Top 20 Categories
top_20_categories = data_no_outliers['product_category'].value_counts().index[:20]
sns.violinplot(x='product_category', y='price', data=data_no_outliers[data_no_outliers['product_category'].isin(top_20_categories)])
plt.xticks(rotation=90)
plt.title("Price Distribution by Product Category (Top 20 Categories)")
plt.show()

# Bar Chart for Average Price of Top 10 Categories
top_10_categories = data_no_outliers['product_category'].value_counts().index[:10]
avg_price_top_10 = data_no_outliers[data_no_outliers['product_category'].isin(top_10_categories)].groupby('product_category')['price'].mean().sort_values(ascending=False)
avg_price_top_10.plot(kind='bar', figsize=(10, 6))
plt.title("Average Price of Products by Top 10 Categories")
plt.ylabel("Average Price")
plt.show()

# Box Plot for Product Ratings Across Categories
sns.boxplot(x='product_category', y='stars', data=data_no_outliers[data_no_outliers['product_category'].isin(top_10_categories)])
plt.xticks(rotation=90)
plt.title("Product Ratings by Category (Top 10 Categories)")
plt.show()

